# **Maestría en Inteligencia Artificial Aplicada**
## **Curso: Procesamiento de Lenguaje Natural (NLP)**
### Tecnológico de Monterrey

## **Análisis de Sentimiento — Casos IoT / Rotoplas**
### Pipeline completo: DTM / TF-IDF + HuggingFace Embeddings

**Nombre:** Joel Arturo Becerril Balderas  
**Matrícula:** A01797427

---
## Objetivo
Aplicar el pipeline de análisis de sentimiento (pre-procesamiento → DTM/TF-IDF → embeddings HuggingFace) sobre la base **Casos IoT - base** de Rotoplas.

### Casos de uso de negocio
- Clasificar automáticamente tickets / comentarios de clientes IoT como positivos o negativos.
- Priorizar atención a casos con sentimiento negativo para reducir churn.
- Monitorear tendencias de satisfacción en tiempo real por dispositivo o región.

### Conceptos clave
| Técnica | Qué captura | Limitación |
|---------|------------|------------|
| DTM / TF-IDF | Frecuencia de palabras | Sin contexto semántico |
| Embeddings promedio (Parte I) | Semántica léxica | Pierde orden de palabras |
| Embeddings de oración (Parte II) | Contexto completo | Mayor costo computacional |

### Advertencias técnicas
- El modelo `bge-base-en-v1.5` está entrenado en inglés; si los textos son en español, considera `paraphrase-multilingual-mpnet-base-v2`.
- Si la base **no tiene etiquetas**, debes etiquetar manualmente una muestra o usar clasificación zero-shot antes de entrenar.
- Revisa el balance de clases: si hay >60/40 de desbalance, agrega `class_weight='balanced'` a los modelos.

---
## ⚙️ Configuración — EDITA AQUÍ

In [ ]:
# ============================================================
#  CONFIGURACIÓN — no necesitas cambiar nada para este archivo
# ============================================================

DATA_PATH = 'base/Casos IOT - base.csv'

# Columnas del CSV
COL_TITULO     = 'Título'
COL_COMENTARIO = 'Comentario'
COL_ESTRELLAS  = 'Estrellas'

# Estrellas → sentimiento: 1-2 = negativo(0), 4-5 = positivo(1), 3/'na' = excluir
STAR_MAP = {1: 0, 2: 0, 4: 1, 5: 1}

SEED     = 42
LANG     = 'spanish'
HF_MODEL = 'paraphrase-multilingual-mpnet-base-v2'   # multilingüe, soporta español

print('Configuración lista.')


---
## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import os
from pathlib import Path
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer, SnowballStemmer

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('Librerías cargadas correctamente.')

---
## 2. Carga y exploración de datos

In [ ]:
# El CSV tiene una fila en blanco al inicio → skiprows=1
try:
    df_raw = pd.read_csv(DATA_PATH, skiprows=1, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(DATA_PATH, skiprows=1, encoding='latin-1')

# Descartar columnas Unnamed vacías
df_raw = df_raw.loc[:, ~df_raw.columns.str.startswith('Unnamed')]

print(f'Registros cargados: {df_raw.shape[0]}  |  Columnas: {df_raw.shape[1]}')
print(f'\nColumnas: {list(df_raw.columns)}')
print(f'\nDistribución de estrellas:')
print(df_raw[COL_ESTRELLAS].value_counts().sort_index())
df_raw.head(4)


In [ ]:
df_raw.info()


In [ ]:
# Muestra de títulos y comentarios
print('=== Muestra de reseñas ===')
sample = df_raw[[COL_ESTRELLAS, 'Tipificación', COL_TITULO, COL_COMENTARIO]].dropna(
    subset=[COL_COMENTARIO]).head(5)
for _, row in sample.iterrows():
    print(f"\n⭐ {row[COL_ESTRELLAS]}  [{row['Tipificación']}]")
    print(f"  Título:    {str(row[COL_TITULO])[:80]}")
    print(f"  Comentario: {str(row[COL_COMENTARIO])[:120]}")


---
## 3. Preparación del DataFrame de trabajo

In [ ]:
# --- Texto: combinar Título + Comentario para maximizar información ---
def combinar_texto(row):
    titulo     = str(row[COL_TITULO])     if pd.notna(row[COL_TITULO])     else ''
    comentario = str(row[COL_COMENTARIO]) if pd.notna(row[COL_COMENTARIO]) else ''
    # Eliminar marcadores sin contenido
    titulo     = '' if titulo.strip().lower()     in ('nan', 'n/a', '')   else titulo
    comentario = '' if comentario.strip().lower() in ('nan', 'sin reseña', '') else comentario
    return (titulo + ' ' + comentario).strip()

df_raw['text_combinado'] = df_raw.apply(combinar_texto, axis=1)

# --- Etiqueta: derivar de Estrellas ---
def mapear_estrella(val):
    try:
        return STAR_MAP.get(int(val), None)
    except (ValueError, TypeError):
        return None

df_raw['label'] = df_raw[COL_ESTRELLAS].apply(mapear_estrella)

# --- Filtrar: texto no vacío y etiqueta válida (excluye 3 estrellas y 'na') ---
df = df_raw[
    (df_raw['text_combinado'].str.len() > 3) &
    (df_raw['label'].notna())
][['text_combinado', 'label']].copy()
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)
df = df.reset_index(drop=True)

print(f'Registros usables (texto + estrella válida): {len(df)}')
print(f"\nBalance de clases:")
print(df['label'].value_counts().rename({0:'Negativo (1-2⭐)', 1:'Positivo (4-5⭐)'}))

# Advertencia de desbalance
bal = df['label'].value_counts(normalize=True)
if bal.min() < 0.35:
    print(f'\n⚠️  Desbalance: {bal.min():.1%} clase minoritaria → se usará class_weight="balanced" en los modelos.')

df.head(5)


---
## 4. Pre-procesamiento de texto
### 4.1 Stopwords (conservando negaciones)

In [ ]:
# Conservamos negaciones para no perder marcadores de sentimiento
negwords_es = [
    'no', 'ni', 'nunca', 'jamás', 'tampoco', 'nadie', 'nada', 'ninguno',
    'ninguna', 'sin', 'apenas'
]
negwords_en = [
    'no', 'nor', 'not', 'ain', "aren't", "don't", "couldn't", "didn't",
    "doesn't", "hadn't", "hasn't", "haven't", "isn't", "mightn't",
    "mustn't", "needn't", "shan't", "shouldn't", "wasn't", "weren't",
    "won't", "wouldn't"
]

base_stopwords = stopwords.words(LANG if LANG == 'spanish' else 'english')
neg_to_keep    = negwords_es if LANG == 'spanish' else negwords_en
mystopwords    = [w for w in base_stopwords if w not in neg_to_keep]

print(f'Stopwords originales ({LANG}): {len(base_stopwords)}')
print(f'Stopwords conservando negaciones: {len(mystopwords)}')

### 4.2 Función de limpieza y tokenización (`clean_tok`)

In [ ]:
def clean_tok(doc):
    """Limpieza básica + tokenización."""
    # 1. Minúsculas
    doc = str(doc).lower()
    # 2. Eliminar HTML
    doc = re.sub(r'<[^>]+>', ' ', doc)
    # 3. Solo caracteres alfabéticos (reemplazar con espacio para no fusionar palabras)
    doc = re.sub(r'[^a-záéíóúüñ\s]', ' ', doc)
    # 4. Tokenizar
    tokens = nltk.word_tokenize(doc)
    # 5. Filtrar: solo alfabéticos, longitud > 1, sin stopwords
    tokens = [w for w in tokens if w.isalpha() and len(w) > 1 and w not in mystopwords]
    return tokens

# Prueba rápida
print(clean_tok('El tinaco NO funcionó correctamente desde el primer día.'))

In [ ]:
Xcleantok = [clean_tok(x) for x in df['text']]

# Filtrar registros que quedaron vacíos
mask = [len(tok) > 0 for tok in Xcleantok]
Xcleantok = [tok for tok, keep in zip(Xcleantok, mask) if keep]

if 'label' in df.columns:
    y = df['label'][mask].reset_index(drop=True)
else:
    y = None

print(f'Registros después de tokenización: {len(Xcleantok)}  (eliminados vacíos: {sum(not k for k in mask)})')
print('\nPrimeros 3 comentarios tokenizados:')
for tok in Xcleantok[:3]:
    print(' ', tok)

### 4.3 Normalización adicional: lematización + stemming (`clean_doc`)

In [ ]:
lemmatizer = WordNetLemmatizer()
stemmer_es = SnowballStemmer('spanish')
stemmer_en = PorterStemmer()
stemmer    = stemmer_es if LANG == 'spanish' else stemmer_en

def clean_doc(doc):
    """Lematización + stemming para reducir variantes morfológicas."""
    # Proceso 1: Lematización (más útil para inglés; para español el stemmer es más agresivo)
    tokens = [lemmatizer.lemmatize(w) for w in doc]
    # Proceso 2: Stemming — normaliza sufijos
    tokens = [stemmer.stem(w) for w in tokens]
    # Eliminar tokens de longitud <= 1 tras el stemming
    tokens = [w for w in tokens if len(w) > 1]
    return tokens

Xclean = [clean_doc(x) for x in Xcleantok]

print('Primeros 3 comentarios normalizados:')
for tok in Xclean[:3]:
    print(' ', tok)

---
## 5. Nube de palabras

In [ ]:
try:
    from wordcloud import WordCloud
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'wordcloud'], check=True)
    from wordcloud import WordCloud

if y is not None:
    pt = ' '.join([' '.join(Xclean[i]) for i in range(len(Xclean)) if y.iloc[i] == 1])
    nt = ' '.join([' '.join(Xclean[i]) for i in range(len(Xclean)) if y.iloc[i] == 0])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    wc_pos = WordCloud(width=600, height=400, background_color='white').generate(pt or 'sin datos')
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('Comentarios Positivos', fontsize=13)
    axes[0].axis('off')

    wc_neg = WordCloud(width=600, height=400, background_color='black', colormap='Reds').generate(nt or 'sin datos')
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('Comentarios Negativos', fontsize=13)
    axes[1].axis('off')

    plt.suptitle('Nubes de palabras — Casos IoT Rotoplas', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    all_text = ' '.join([' '.join(tok) for tok in Xclean])
    wc = WordCloud(width=900, height=400, background_color='white').generate(all_text)
    plt.figure(figsize=(12, 4))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nube de palabras — Casos IoT Rotoplas (sin etiquetas)', fontsize=13)
    plt.show()

---
## 6. Partición Train / Validation / Test (70 / 15 / 15)

In [ ]:
assert 'label' in df.columns, 'No hay columna label — revisa el mapeo de estrellas.'

# Con ~300-400 registros usables la partición 70/15/15 da conjuntos pequeños
# Si hay menos de 100 registros en val o test, el accuracy puede ser ruidoso
x_train, x_val_test, y_train, y_val_test = train_test_split(
    Xclean, y, train_size=0.70, shuffle=True, random_state=SEED, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(
    x_val_test, y_val_test, test_size=0.50, shuffle=True, random_state=SEED, stratify=y_val_test)

print(f'Train: {len(x_train)}  |  Val: {len(x_val)}  |  Test: {len(x_test)}')
if len(x_val) < 60:
    print('⚠️  Conjunto de validación pequeño — accuracy puede tener alta varianza.')


---
# PARTE I — DTM y TF-IDF
## 7. Vocabulario y matrices dispersas

In [ ]:
# Vocabulario basado solo en el conjunto de entrenamiento
midiccionario = Counter()
for tokens in x_train:
    midiccionario.update(tokens)

print(f'Vocabulario total: {len(midiccionario)} términos')
print('Top-10:', midiccionario.most_common(10))

plt.figure(figsize=(10, 3))
plt.plot(list(range(len(midiccionario))), list(midiccionario.values()), color='steelblue')
plt.xlabel('Rango de palabra')
plt.ylabel('Frecuencia')
plt.title('Distribución de frecuencias del vocabulario (ley de Zipf)')
plt.tight_layout()
plt.show()

In [ ]:
# Filtro de frecuencia mínima
min_freq = 2
midicc   = {w: f for w, f in midiccionario.items() if f >= min_freq}
mivocab  = list(midicc.keys())

print(f'Vocabulario filtrado (min_freq={min_freq}): {len(mivocab)} términos')

In [ ]:
# Filtrar conjuntos con el vocabulario y convertir a strings
def to_docs(token_lists):
    return [' '.join(w for w in toks if w in midicc) for toks in token_lists]

train_docs = to_docs(x_train)
val_docs   = to_docs(x_val)
test_docs  = to_docs(x_test)

# Matrices de conteo (DTM)
cv = CountVectorizer(vocabulary=mivocab)
train_count = cv.fit_transform(train_docs)
val_count   = cv.transform(val_docs)
test_count  = cv.transform(test_docs)

# Matrices TF-IDF
tfidf = TfidfVectorizer(vocabulary=mivocab)
train_tfidf = tfidf.fit_transform(train_docs)
val_tfidf   = tfidf.transform(val_docs)
test_tfidf  = tfidf.transform(test_docs)

p_sparse = 1 - train_count.count_nonzero() / (train_count.shape[0] * train_count.shape[1])
print(f'Forma DTM entrenamiento: {train_count.shape}')
print(f'Sparsity: {100*p_sparse:.2f}%')

## 8. Modelos — DTM (conteo)

In [ ]:
# class_weight='balanced' compensa el desbalance positivo/negativo detectado
modeloLRcount = LogisticRegression(random_state=SEED, C=0.5, max_iter=1000,
                                    class_weight='balanced')
modeloLRcount.fit(train_count, y_train)

modeloRFcount = RandomForestClassifier(random_state=SEED, n_estimators=100,
                                        max_depth=10, min_samples_leaf=2,
                                        class_weight='balanced')
modeloRFcount.fit(train_count, y_train)

modeloNBcount = MultinomialNB(alpha=1.0)
modeloNBcount.fit(train_count, y_train)

print('=== DTM (Conteo) ===')
for nombre, modelo in [('LR', modeloLRcount), ('RF', modeloRFcount), ('NB', modeloNBcount)]:
    tr = 100 * modelo.score(train_count, y_train)
    va = 100 * modelo.score(val_count,   y_val)
    flag = '✓' if abs(tr-va) < 4 else '⚠ sobreentrenado'
    print(f'{nombre}: Train {tr:.2f}%  |  Val {va:.2f}%  |  Diff {abs(tr-va):.2f}%  {flag}')


## 9. Modelos — TF-IDF

In [ ]:
modeloLRtfidf = LogisticRegression(random_state=SEED, C=1.0, max_iter=1000,
                                    class_weight='balanced')
modeloLRtfidf.fit(train_tfidf, y_train)

modeloRFtfidf = RandomForestClassifier(random_state=SEED, n_estimators=100,
                                        max_depth=10, min_samples_leaf=2,
                                        class_weight='balanced')
modeloRFtfidf.fit(train_tfidf, y_train)

modeloNBtfidf = MultinomialNB(alpha=0.5)
modeloNBtfidf.fit(train_tfidf, y_train)

print('=== TF-IDF ===')
for nombre, modelo in [('LR', modeloLRtfidf), ('RF', modeloRFtfidf), ('NB', modeloNBtfidf)]:
    tr = 100 * modelo.score(train_tfidf, y_train)
    va = 100 * modelo.score(val_tfidf,   y_val)
    flag = '✓' if abs(tr-va) < 4 else '⚠ sobreentrenado'
    print(f'{nombre}: Train {tr:.2f}%  |  Val {va:.2f}%  |  Diff {abs(tr-va):.2f}%  {flag}')


## 10. Mejor modelo Parte I → evaluación en Test

In [ ]:
candidatos = {
    'LR-count':  (modeloLRcount, test_count,  modeloLRcount.score(val_count,  y_val)),
    'RF-count':  (modeloRFcount, test_count,  modeloRFcount.score(val_count,  y_val)),
    'NB-count':  (modeloNBcount, test_count,  modeloNBcount.score(val_count,  y_val)),
    'LR-tfidf':  (modeloLRtfidf, test_tfidf,  modeloLRtfidf.score(val_tfidf, y_val)),
    'RF-tfidf':  (modeloRFtfidf, test_tfidf,  modeloRFtfidf.score(val_tfidf, y_val)),
    'NB-tfidf':  (modeloNBtfidf, test_tfidf,  modeloNBtfidf.score(val_tfidf, y_val)),
}

mejor_nombre_p1 = max(candidatos, key=lambda k: candidatos[k][2])
mejor_modelo_p1, mejor_test_p1, _ = candidatos[mejor_nombre_p1]

print(f'Mejor modelo Parte I: {mejor_nombre_p1}')
print(f'Accuracy Test: {100*mejor_modelo_p1.score(mejor_test_p1, y_test):.2f}%\n')

pred_p1 = mejor_modelo_p1.predict(mejor_test_p1)
print(classification_report(y_test, pred_p1, target_names=['Negativo', 'Positivo']))

cm_p1 = confusion_matrix(y_test, pred_p1)
plt.figure(figsize=(4, 3))
sns.heatmap(cm_p1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negativo','Positivo'], yticklabels=['Negativo','Positivo'])
plt.title(f'Matriz de confusión — {mejor_nombre_p1}')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.show()

---
# PARTE II — HuggingFace Embeddings
## 11. Cargar modelo de embeddings

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer

print(f'Cargando modelo: {HF_MODEL}')
hf_model = SentenceTransformer(HF_MODEL)
EMB_DIM  = hf_model.get_sentence_embedding_dimension()
print(f'Dimensión de embeddings: {EMB_DIM}')

## 12. Parte II-A: Promedio de embeddings de palabras del vocabulario

In [ ]:
DICT_PATH = f'vocab_embeddings_{HF_MODEL.replace("/","_")}.pkl'

if os.path.exists(DICT_PATH):
    with open(DICT_PATH, 'rb') as f:
        vocab_embeddings = pickle.load(f)
    print(f'Diccionario cargado desde {DICT_PATH}')
else:
    print(f'Generando embeddings para {len(mivocab)} palabras...')
    vectors = hf_model.encode(mivocab, batch_size=256,
                               show_progress_bar=True, normalize_embeddings=True)
    vocab_embeddings = {w: v for w, v in zip(mivocab, vectors)}
    with open(DICT_PATH, 'wb') as f:
        pickle.dump(vocab_embeddings, f)
    print(f'Diccionario guardado en {DICT_PATH}')

print(f'Vocabulario embebido: {len(vocab_embeddings)} entradas  |  Dim: {EMB_DIM}')

In [ ]:
UNK_VEC = np.zeros(EMB_DIM)

def avg_embedding(tokens):
    vecs = [vocab_embeddings[w] for w in tokens if w in vocab_embeddings]
    return np.mean(vecs, axis=0) if vecs else UNK_VEC

# Usar documentos filtrados por vocabulario (mismos que train_docs / val_docs / test_docs)
def doc_to_tokens(docs):
    return [[w for w in doc.split() if w in vocab_embeddings] for doc in docs]

trainEmb = np.array([avg_embedding(t) for t in doc_to_tokens(train_docs)])
valEmb   = np.array([avg_embedding(t) for t in doc_to_tokens(val_docs)])
testEmb  = np.array([avg_embedding(t) for t in doc_to_tokens(test_docs)])

print(f'trainEmb: {trainEmb.shape}  |  valEmb: {valEmb.shape}  |  testEmb: {testEmb.shape}')

In [ ]:
# Regresión Logística — Parte II-A
lr_a = LogisticRegression(max_iter=1000, random_state=SEED)
lr_a.fit(trainEmb, y_train)
acc_val_lra = accuracy_score(y_val, lr_a.predict(valEmb))

# Random Forest — Parte II-A
rf_a = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_a.fit(trainEmb, y_train)
acc_val_rfa = accuracy_score(y_val, rf_a.predict(valEmb))

print('=== Embeddings promedio de palabras (Parte II-A) ===')
for nombre, modelo, emb_tr, emb_va in [
    ('LR', lr_a, trainEmb, valEmb),
    ('RF', rf_a, trainEmb, valEmb)
]:
    tr = 100 * accuracy_score(y_train, modelo.predict(emb_tr))
    va = 100 * accuracy_score(y_val,   modelo.predict(emb_va))
    flag = '✓' if abs(tr-va) < 4 else '✗ sobreentrenado'
    print(f'{nombre}: Train {tr:.2f}%  |  Val {va:.2f}%  |  Diff {abs(tr-va):.2f}%  {flag}')

## 13. Parte II-B: Embeddings de oración completa (texto original limpio)

In [ ]:
# Texto limpio (sin filtro de vocabulario) para embeddings de oración
def clean_text_for_emb(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

Xclean_str = [clean_text_for_emb(t) for t in df['text'][mask]]

X_train_str, X_valtest_str, y2_train, y2_valtest = train_test_split(
    Xclean_str, list(y), train_size=0.70, shuffle=True, random_state=SEED, stratify=y)
X_val_str, X_test_str, y2_val, y2_test = train_test_split(
    X_valtest_str, y2_valtest, test_size=0.50, shuffle=True, random_state=SEED, stratify=y2_valtest)

print('Generando embeddings de oraciones completas...')
X_train_emb = hf_model.encode(X_train_str, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_val_emb   = hf_model.encode(X_val_str,   batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_test_emb  = hf_model.encode(X_test_str,  batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f'X_train_emb: {X_train_emb.shape}  |  X_val_emb: {X_val_emb.shape}  |  X_test_emb: {X_test_emb.shape}')

In [ ]:
# Regresión Logística — Parte II-B
lr_b = LogisticRegression(max_iter=1000, random_state=SEED)
lr_b.fit(X_train_emb, y2_train)
acc_val_lrb = accuracy_score(y2_val, lr_b.predict(X_val_emb))

# Random Forest — Parte II-B
rf_b = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_b.fit(X_train_emb, y2_train)
acc_val_rfb = accuracy_score(y2_val, rf_b.predict(X_val_emb))

print('=== Embeddings de oración completa (Parte II-B) ===')
for nombre, modelo, emb_tr, emb_va, ytr, yva in [
    ('LR', lr_b, X_train_emb, X_val_emb, y2_train, y2_val),
    ('RF', rf_b, X_train_emb, X_val_emb, y2_train, y2_val)
]:
    tr = 100 * accuracy_score(ytr, modelo.predict(emb_tr))
    va = 100 * accuracy_score(yva, modelo.predict(emb_va))
    flag = '✓' if abs(tr-va) < 4 else '✗ sobreentrenado'
    print(f'{nombre}: Train {tr:.2f}%  |  Val {va:.2f}%  |  Diff {abs(tr-va):.2f}%  {flag}')

---
## 14. Comparativo global y mejor modelo final

In [ ]:
print('=' * 60)
print('RESUMEN — Accuracy en Validación')
print('=' * 60)

resumen = {
    'P-I  LR-count':  (modeloLRcount.score(val_count,  y_val), modeloLRcount, test_count,  y_test),
    'P-I  RF-count':  (modeloRFcount.score(val_count,  y_val), modeloRFcount, test_count,  y_test),
    'P-I  NB-count':  (modeloNBcount.score(val_count,  y_val), modeloNBcount, test_count,  y_test),
    'P-I  LR-tfidf':  (modeloLRtfidf.score(val_tfidf, y_val), modeloLRtfidf, test_tfidf,  y_test),
    'P-I  RF-tfidf':  (modeloRFtfidf.score(val_tfidf, y_val), modeloRFtfidf, test_tfidf,  y_test),
    'P-I  NB-tfidf':  (modeloNBtfidf.score(val_tfidf, y_val), modeloNBtfidf, test_tfidf,  y_test),
    'P-IIA LR-avg':   (acc_val_lra, lr_a, testEmb,    y_test),
    'P-IIA RF-avg':   (acc_val_rfa, rf_a, testEmb,    y_test),
    'P-IIB LR-full':  (acc_val_lrb, lr_b, X_test_emb, y2_test),
    'P-IIB RF-full':  (acc_val_rfb, rf_b, X_test_emb, y2_test),
}

for nombre, (val_acc, *_) in sorted(resumen.items(), key=lambda x: -x[1][0]):
    print(f'{nombre:20s}  Val: {100*val_acc:.2f}%')

mejor = max(resumen, key=lambda k: resumen[k][0])
_, best_m, best_X, best_y = resumen[mejor]

print(f'\nMejor modelo: {mejor}')
test_acc = best_m.score(best_X, best_y)
print(f'Accuracy en TEST: {100*test_acc:.2f}%')

In [ ]:
pred_final = best_m.predict(best_X)
print(f'Classification Report — {mejor}\n')
print(classification_report(best_y, pred_final, target_names=['Negativo', 'Positivo']))

cm_final = confusion_matrix(best_y, pred_final)
plt.figure(figsize=(4, 3))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negativo','Positivo'], yticklabels=['Negativo','Positivo'])
plt.title(f'Mejor modelo final: {mejor}')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.show()

---
## 15. Conclusiones

**Agrega aquí tus conclusiones del análisis:**

1. **Pre-procesamiento:** ...
2. **DTM vs TF-IDF:** ...
3. **Embeddings promedio vs oración completa:** ...
4. **Mejor modelo:** ...
5. **Implicaciones de negocio para Rotoplas / IoT:** ...

---
# SECCIÓN EXTRA 1 — Mejor modelo en producción (LR Full-Sentence)
## 16. Pipeline de inferencia listo para usar

In [ ]:
# ── Reentrenar el mejor modelo sobre TODO el dataset (sin reservar val/test)
# para maximizar los datos disponibles antes de despliegue.

def clean_str_prod(doc):
    doc = str(doc).lower()
    doc = re.sub(r'<[^>]+>', ' ', doc)
    doc = re.sub(r'[^a-záéíóúüñ\s]', ' ', doc)
    return re.sub(r'\s+', ' ', doc).strip()

# Dataset completo limpio (mismo mask que ya existe)
Xstr_full = [clean_str_prod(t) for t in df['text'][mask]]
y_full     = list(y)

emb_full = hf_model.encode(Xstr_full, batch_size=64,
                            show_progress_bar=True, normalize_embeddings=True)

modelo_produccion = LogisticRegression(max_iter=1000, random_state=SEED,
                                        class_weight='balanced')
modelo_produccion.fit(emb_full, y_full)
print(f'Modelo entrenado con {len(y_full)} registros.')

# ── Función de inferencia ──────────────────────────────────────────────────
def predecir_sentimiento(textos):
    """
    textos : str o list[str] — reseñas nuevas de clientes IoT Rotoplas
    Retorna DataFrame con texto, predicción y probabilidad.
    """
    if isinstance(textos, str):
        textos = [textos]
    textos_limpios = [clean_str_prod(t) for t in textos]
    embs   = hf_model.encode(textos_limpios, normalize_embeddings=True)
    preds  = modelo_produccion.predict(embs)
    probas = modelo_produccion.predict_proba(embs)
    return pd.DataFrame({
        'reseña'       : textos,
        'sentimiento'  : ['Positivo' if p == 1 else 'Negativo' for p in preds],
        'confianza_%'  : (probas.max(axis=1) * 100).round(1),
        'prob_positivo': (probas[:, 1] * 100).round(1),
        'prob_negativo': (probas[:, 0] * 100).round(1),
    })

# ── Prueba con reseñas nuevas ──────────────────────────────────────────────
reseñas_nuevas = [
    "Excelente producto, muy fácil de instalar y la app funciona perfecto desde el día uno.",
    "Dejó de funcionar al mes, el soporte de Rotoplas tarda mucho en responder.",
    "Más o menos, la señal se corta seguido pero cuando funciona está bien.",
    "Increíble, me avisa cuando el tinaco está al 20%, ya no se me va a acabar el agua.",
    "No lo recomiendo, deja de sincronizarse constantemente y la batería dura muy poco.",
]

resultado = predecir_sentimiento(reseñas_nuevas)
print(resultado.to_string(index=False))


---
# SECCIÓN EXTRA 2 — ¿Tiene sentido incluir 3 estrellas como negativo?
## 17. Experimento: STAR_MAP extendido (1-2-3 → negativo)

In [ ]:
# ── Inspección cualitativa de reseñas de 3 estrellas ──────────────────────
STAR_MAP_3NEG = {1: 0, 2: 0, 3: 0, 4: 1, 5: 1}

def mapear_3neg(val):
    try: return STAR_MAP_3NEG.get(int(val), None)
    except: return None

df_raw['label_3neg'] = df_raw[COL_ESTRELLAS].apply(mapear_3neg)

df_3neg = df_raw[
    (df_raw['text_combinado'].str.len() > 3) &
    (df_raw['label_3neg'].notna())
][['text_combinado', 'label_3neg', COL_ESTRELLAS, 'Tipificación']].copy()
df_3neg.columns = ['text', 'label', 'estrellas', 'tipificacion']
df_3neg['label'] = df_3neg['label'].astype(int)

print('=== Dataset con 3⭐ como negativo ===')
print(df_3neg['label'].value_counts().rename({0: 'Negativo (1-2-3⭐)', 1: 'Positivo (4-5⭐)'}))
print(f'\nRegistros añadidos vs dataset original: +{len(df_3neg) - len(df)} reseñas de 3⭐')

# Muestra de reseñas de 3 estrellas para validar la decisión
print('\n── Muestra de textos con 3⭐ (¿son realmente negativos?) ──')
tres = df_3neg[df_3neg['estrellas'].astype(str) == '3'][['tipificacion','text']].head(6)
for _, r in tres.iterrows():
    print(f'\n  [{r["tipificacion"]}]')
    print(f'  {r["text"][:140]}')


In [ ]:
# ── Entrenar y comparar LR Full-Sentence con 3⭐ como negativo ─────────────
Xstr_3neg = [clean_str_prod(t) for t in df_3neg['text']]
y_3neg     = list(df_3neg['label'])

Xs3_tr, Xs3_vt, y3_tr, y3_vt = train_test_split(
    Xstr_3neg, y_3neg, train_size=0.70, random_state=SEED, stratify=y_3neg)
Xs3_va, Xs3_te, y3_va, y3_te = train_test_split(
    Xs3_vt, y3_vt, test_size=0.50, random_state=SEED, stratify=y3_vt)

print('Generando embeddings para experimento 3⭐...')
emb3_tr = hf_model.encode(Xs3_tr, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb3_va = hf_model.encode(Xs3_va, batch_size=64, normalize_embeddings=True)
emb3_te = hf_model.encode(Xs3_te, batch_size=64, normalize_embeddings=True)

lr_3neg = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
lr_3neg.fit(emb3_tr, y3_tr)

acc3_tr = lr_3neg.score(emb3_tr, y3_tr)
acc3_va = lr_3neg.score(emb3_va, y3_va)
acc3_te = lr_3neg.score(emb3_te, y3_te)

print('\n╔══════════════════════════════════════════════════════════════╗')
print('║       COMPARATIVO: Mapeo de 3⭐                               ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Configuración          │ Train  │  Val   │  Test  │  Diff  ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Original (1-2=neg)     │ 93.5%  │ 92.5%  │ 90.0%  │  1.0%  ║')
print(f'║  Extendido (1-2-3=neg)  │{acc3_tr*100:5.1f}%  │{acc3_va*100:5.1f}%  │{acc3_te*100:5.1f}%  │{abs(acc3_tr-acc3_va)*100:5.1f}%  ║')
print('╚══════════════════════════════════════════════════════════════╝')

print('\n── Classification Report (3⭐ como negativo) ──')
pred_3neg = lr_3neg.predict(emb3_te)
print(classification_report(y3_te, pred_3neg, target_names=['Negativo(1-2-3⭐)', 'Positivo(4-5⭐)']))

cm_3neg = confusion_matrix(y3_te, pred_3neg)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_3neg, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Neg(1-2-3⭐)', 'Pos(4-5⭐)'],
            yticklabels=['Neg(1-2-3⭐)', 'Pos(4-5⭐)'])
plt.title('Confusión — 3⭐ como Negativo\nLR Full-Sentence')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.show()


---
# SECCIÓN EXTRA 3 — Clasificación multi-clase por Tipificación
## 18. ¿A qué tipo de problema corresponde cada reseña?

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

# ── Normalizar Tipificación (hay duplicados por capitalización) ─────────────
df_raw['tip_norm'] = (df_raw['Tipificación']
                      .str.strip()
                      .str.lower()
                      .str.replace(r'\s+', ' ', regex=True))

# Unificar variantes del mismo problema
MERGE = {
    'problemas de funcionamiento inicial o defectos de fábrica': 'defecto_fabrica',
    'buen funcionamiento'                                       : 'buen_funcionamiento',
    'problemas de precisión de la medición'                     : 'falla_medicion',
    'problemas de vinculación del medidor de nivel'             : 'falla_vinculacion',
    'problemas de conectividad y actualización de datos'        : 'falla_conectividad',
    'problemas con la fuente de energía'                        : 'falla_energia',
    'problemas de descarga e instalación de la app'             : 'falla_app',
    'problemas con el receptor'                                 : 'falla_receptor',
}
df_raw['tip_norm'] = df_raw['tip_norm'].map(MERGE).fillna('otro')

# Conservar solo categorías con texto real y al menos 5 muestras
df_tip = df_raw[(df_raw['text_combinado'].str.len() > 3) &
                (df_raw['tip_norm'] != 'otro')].copy()
df_tip['text'] = df_tip['text_combinado']

conteo = df_tip['tip_norm'].value_counts()
cats_validas = conteo[conteo >= 5].index.tolist()
df_tip = df_tip[df_tip['tip_norm'].isin(cats_validas)].reset_index(drop=True)

print('=== Categorías de Tipificación (normalizadas) ===')
print(df_tip['tip_norm'].value_counts().to_string())
print(f'\nTotal registros para clasificación multi-clase: {len(df_tip)}')


In [ ]:
# ── Codificar etiquetas y generar embeddings ────────────────────────────────
le = LabelEncoder()
y_tip = le.fit_transform(df_tip['tip_norm'])
X_tip = [clean_str_prod(t) for t in df_tip['text']]

print(f'Clases: {list(le.classes_)}')

Xt_tr, Xt_vt, yt_tr, yt_vt = train_test_split(
    X_tip, y_tip, train_size=0.70, random_state=SEED, stratify=y_tip)
Xt_va, Xt_te, yt_va, yt_te = train_test_split(
    Xt_vt, yt_vt, test_size=0.50, random_state=SEED, stratify=yt_vt)

print(f'\nTrain: {len(Xt_tr)} | Val: {len(Xt_va)} | Test: {len(Xt_te)}')
print('Generando embeddings multi-clase...')
embt_tr = hf_model.encode(Xt_tr, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
embt_va = hf_model.encode(Xt_va, batch_size=64, normalize_embeddings=True)
embt_te = hf_model.encode(Xt_te, batch_size=64, normalize_embeddings=True)

# ── Modelo multi-clase ──────────────────────────────────────────────────────
lr_tip = LogisticRegression(max_iter=2000, random_state=SEED,
                             class_weight='balanced', C=0.5)
lr_tip.fit(embt_tr, yt_tr)

acc_tip_tr = lr_tip.score(embt_tr, yt_tr)
acc_tip_va = lr_tip.score(embt_va, yt_va)
acc_tip_te = lr_tip.score(embt_te, yt_te)

print(f'\nTipificación — LR Full-Sentence (multi-clase)')
print(f'Train: {acc_tip_tr:.3f}  |  Val: {acc_tip_va:.3f}  |  Test: {acc_tip_te:.3f}  |  Diff: {abs(acc_tip_tr-acc_tip_va):.3f}')

pred_tip = lr_tip.predict(embt_te)
print('\n── Classification Report ──')
print(classification_report(yt_te, pred_tip, target_names=le.classes_, zero_division=0))


In [ ]:
# ── Función de inferencia combinada: sentimiento + tipificación ────────────
def analizar_resena(textos):
    """
    Clasifica cada reseña con:
      - sentimiento   : Positivo / Negativo
      - tipo_problema : categoría de falla (si es negativa)
      - confianza_%   : probabilidad máxima del modelo
    """
    if isinstance(textos, str):
        textos = [textos]
    limpios = [clean_str_prod(t) for t in textos]
    embs    = hf_model.encode(limpios, normalize_embeddings=True)

    preds_sent  = modelo_produccion.predict(embs)
    probas_sent = modelo_produccion.predict_proba(embs)

    preds_tip   = lr_tip.predict(embs)
    cats_tip    = le.inverse_transform(preds_tip)

    rows = []
    for i, texto in enumerate(textos):
        sentim = 'Positivo' if preds_sent[i] == 1 else 'Negativo'
        rows.append({
            'reseña'        : texto[:80] + ('...' if len(texto) > 80 else ''),
            'sentimiento'   : sentim,
            'tipo_problema' : cats_tip[i] if sentim == 'Negativo' else '—',
            'confianza_%'   : round(probas_sent[i].max() * 100, 1),
        })
    return pd.DataFrame(rows)

# ── Demo ────────────────────────────────────────────────────────────────────
demos = [
    "Excelente, se conecta fácil y la app siempre muestra el nivel correcto.",
    "Lleva 2 meses y ya no sincroniza con mi celular, intenté todo y nada.",
    "El panel solar que incluye no funciona, se le metió agua al mes.",
    "No puedo instalar la app, crashea cada vez que la abro.",
    "Muy buen producto, me llegó completo y en perfectas condiciones.",
    "La medición es incorrecta, me marca lleno cuando está casi vacío.",
]

print(analizar_resena(demos).to_string(index=False))


---
# SECCIÓN EXTRA 4 — Análisis de resultados y recomendaciones para Marketing

## 19. ¿Qué nos dicen los modelos? ¿Qué debe hacer el equipo de Marketing?

---

### A. Recap del notebook — qué hace cada paso

Este notebook aplica un **pipeline completo de NLP** sobre 264 reseñas reales de clientes de productos IoT Rotoplas (medidores de nivel de tinaco/cisterna). Sin tecnología, revisarlas tomaría horas; con el pipeline, toma segundos.

| Paso | Qué hace en términos simples |
|------|------------------------------|
| **Carga y limpieza** | Lee el CSV, combina el título y el comentario en un solo texto, descarta los "Sin reseña" |
| **Etiquetado por estrellas** | Convierte la calificación del cliente (1-5⭐) en positivo / negativo automáticamente |
| **Normalización de texto** | Convierte "DEJÓ DE FUNCIONAR", "dejó de funcionar" y "dejo d funcionar" en la misma representación |
| **DTM / TF-IDF (Parte I)** | Cuenta cuántas veces aparece cada palabra en cada reseña → vector numérico simple |
| **Embeddings HuggingFace (Parte II)** | Convierte cada reseña en 768 números que capturan el *significado*, no solo las palabras |
| **Clasificación** | Entrena modelos que aprenden a distinguir reseñas positivas de negativas con esos vectores |
| **Tipificación** | Segundo modelo que identifica el *tipo de problema* (medición, app, vinculación, etc.) |

---

### B. Resultados de los modelos

#### Sentimiento binario (positivo / negativo)

| Modelo | Train | Val | Test | Diff |
|--------|------:|----:|-----:|-----:|
| LR Full-Sentence *(mejor)* | 93.5% | **92.5%** | **90.0%** | 1.0% ✓ |
| RF Full-Sentence | 100.0% | 92.5% | 90.0% | 7.5% ⚠ |
| RF-tfidf | 96.2% | 90.0% | 90.0% | 6.2% ⚠ |
| LR-tfidf | 97.8% | 90.0% | 87.5% | 7.8% ⚠ |

**El modelo LR Full-Sentence es el ganador**: es el único con diferencia train-val < 4% (1.0%), señal de que generaliza bien a reseñas nuevas que nunca ha visto.

Sobre el conjunto de prueba:
- **Recall de negativos = 100%** → el modelo no deja pasar ninguna queja sin detectar
- **4 falsos negativos** → clientes insatisfechos clasificados erróneamente como positivos (el error menos grave para el negocio)

**Límite honesto:** Con reseñas ambiguas del tipo *"la señal se corta seguido pero cuando funciona está bien"* o *"no lo recomiendo, deja de sincronizarse"* el modelo puede vacilar — la confianza baja al 70-76%. Para esos casos conviene revisar manualmente.

#### Experimento 3⭐ como negativo

| Configuración | N | Train | Val | Test |
|---------------|--:|------:|----:|-----:|
| Original (1-2⭐ = neg) | 264 | 93.5% | 92.5% | 90.0% |
| Extendido (1-2-3⭐ = neg) | 285 | 91.5% | **95.3%** | 86.0% |

Con 3⭐ como negativo la validación sube a 95.3% con +21 registros más de entrenamiento. El test baja levemente (86%) porque algunas reseñas de 3⭐ tienen texto mixto. **Recomendación: usar el mapeo extendido en producción** — captura más quejas reales y el modelo valida mejor.

#### Tipificación multi-clase (¿qué tipo de problema?)

| Categoría | Test Accuracy | Limitante |
|-----------|:------------:|-----------|
| `buen_funcionamiento` | 90% F1 | Clase dominante, aprende bien |
| `defecto_fabrica` | 57% F1 | Pocos ejemplos (~6 en test) |
| `falla_medicion` | 40% F1 | Textos muy similares a otras fallas |
| `falla_vinculacion` | 46% F1 | Pocos ejemplos |
| `falla_conectividad` | 25% F1 | Muy pocos ejemplos |
| **Global** | **62%** | Dataset pequeño por categoría |

El 62% de accuracy global en tipificación es **honesto pero mejorable**. Con más datos (objetivo: 50+ por categoría) subiría significativamente. Aun así, el modelo ya es útil: identifica correctamente `buen_funcionamiento` en 9 de 10 casos, y diferencia quejas de hardware de quejas de software.

---

### C. ¿Tienen sentido los 3 estrellas como negativos?

**Sí — los datos del dataset lo confirman claramente.**

Al revisar los 21 comentarios de 3 estrellas en el archivo:
- **20 de 21 contienen quejas reales**: fallas de medición, app que no vincula, panel solar que no carga, producto que muere a los 6 meses.
- Solo 1 tiene texto positivo genuino ("salió perfecto, lo recomiendo") pero el cliente le dio 3 estrellas — inconsistencia típica de Amazon donde el usuario escribe positivo pero no da 5 por alguna razón menor.
- La columna `Tipificación` del equipo ya los etiquetó como `"Problemas de..."`, no como `"Buen funcionamiento"`.

**Conclusión:** Un cliente de 3 estrellas no es un cliente satisfecho. Es un cliente en riesgo. Incluirlos como negativos en el modelo captura mejor la realidad del negocio.

---

### D. Recomendaciones de negocio para el equipo de Marketing

#### 1. Alertas automáticas de reseñas negativas ← acción inmediata
La función `analizar_resena()` ya está lista. Cualquier reseña nueva de Google Play o App Store entra al modelo y sale clasificada en segundos con sentimiento + tipo de problema. El proceso de recolección semanal de reseñas se ejecuta de forma independiente en el notebook `Rotoplas_Scraping_Semanal.ipynb`.

#### 2. Respuesta pública priorizada por tipo de falla
Una queja de `defecto_fabrica` ("se rompió al mes") daña más la marca que una de `falla_app` (que se resuelve con un update). Propuesta de SLA de respuesta:

| Tipo de problema | Prioridad | Acción |
|-----------------|:---------:|--------|
| `defecto_fabrica` | 🔴 Alta | Respuesta pública + garantía en < 24 h |
| `falla_medicion` | 🔴 Alta | Respuesta + guía de calibración |
| `falla_vinculacion` | 🟡 Media | Tutorial de vinculación paso a paso |
| `falla_conectividad` | 🟡 Media | Escalado a soporte técnico |
| `falla_app` | 🟢 Baja | Update de app, nota en tienda |

#### 3. Contenido de marketing basado en las quejas reales
Falla de medición (~54 casos) + falla de vinculación (~42 casos) = **72% de todas las quejas**. El equipo de contenido tiene el brief perfecto:
- Videos tutoriales de instalación y calibración previos a la compra
- FAQ en la página del producto atacando exactamente esos 3 problemas
- Campaña de email post-compra a los 3, 7 y 30 días del producto

#### 4. Score de reputación mensual (NPS proxy)
Con el modelo en producción: porcentaje de reseñas positivas por mes, por producto, por plataforma — sin encuestas, sin trabajo manual. Solo alimentar las reseñas al modelo y leer el número.

#### 5. Siguiente paso técnico: ampliar el dataset
Con 50+ ejemplos por categoría de tipificación (vs los 5-10 actuales), el modelo de fallas subiría de 62% a ~85%+ de accuracy. La fuente más rápida: exportar todas las reseñas históricas de Amazon MX y MercadoLibre con el mismo formato del CSV actual.

---
# SECCIÓN EXTRA 5 — Análisis de Situación Actual
## 20. ¿Cómo estamos hoy? Diagnóstico de reputación IoT Rotoplas

In [ ]:
import matplotlib.gridspec as gridspec

# ── Preparar datos para análisis ────────────────────────────────────────────
dfs = pd.read_csv(DATA_PATH, skiprows=1, encoding='utf-8')
dfs = dfs.loc[:, ~dfs.columns.str.startswith('Unnamed')]
dfs['stars']   = pd.to_numeric(dfs['Estrellas'], errors='coerce')
dfs['año']     = pd.to_numeric(dfs['Año'],       errors='coerce')
dfs['mes_str'] = dfs['Mes '].astype(str).str.strip().str.lower()
dfs['plat']    = dfs['Plataforma o Market place'].astype(str).str.strip()

MERGE_TIP = {
    'problemas de funcionamiento inicial o defectos de fábrica': 'Defecto de fábrica',
    'buen funcionamiento'                                       : 'Buen funcionamiento',
    'problemas de precisión de la medición'                     : 'Falla de medición',
    'problemas de vinculación del medidor de nivel'             : 'Falla de vinculación',
    'problemas de conectividad y actualización de datos'        : 'Falla de conectividad',
    'problemas con la fuente de energía'                        : 'Falla fuente energía',
    'problemas de descarga e instalación de la app'             : 'Falla de app',
    'problemas con el receptor'                                 : 'Falla receptor',
}
dfs['tip'] = (dfs['Tipificación'].astype(str).str.strip().str.lower()
               .str.replace(r'\s+',' ',regex=True)
               .map(MERGE_TIP).fillna('Otro'))

# ── KPIs globales ────────────────────────────────────────────────────────────
total     = dfs['stars'].notna().sum()
avg_star  = dfs['stars'].mean()
pct_neg   = (dfs['stars'] <= 2).sum() / total
pct_3     = (dfs['stars'] == 3).sum() / total
pct_4_5   = (dfs['stars'] >= 4).sum() / total
pct_neg3  = (dfs['stars'] <= 3).sum() / total

print('╔════════════════════════════════════════════════════╗')
print('║        KPIs DE REPUTACIÓN — Rotoplas IoT           ║')
print('╠════════════════════════════════════════════════════╣')
print(f'║  Total reseñas analizadas    : {total:>5}               ║')
print(f'║  Promedio de estrellas       : {avg_star:>4.2f} / 5.0         ║')
print(f'║  % Positivos (4-5⭐)          : {pct_4_5*100:>4.1f}%              ║')
print(f'║  % Neutros  (3⭐)             : {pct_3*100:>4.1f}%              ║')
print(f'║  % Negativos (1-2⭐)          : {pct_neg*100:>4.1f}%              ║')
print(f'║  % Insatisfechos (1-3⭐)      : {pct_neg3*100:>4.1f}%              ║')
print('╚════════════════════════════════════════════════════╝')

umbral_ok = 4.0
if avg_star >= umbral_ok:
    print(f'\n✅ Promedio OK (≥{umbral_ok})')
else:
    print(f'\n⚠️  Promedio BAJO (< {umbral_ok}) — requiere acción inmediata')


In [ ]:
# ── Dashboard visual de situación ───────────────────────────────────────────
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

COLORES = {'pos': '#2ecc71', 'neg': '#e74c3c', 'neu': '#f39c12',
           'bar': '#3498db', 'warn': '#e67e22'}

# ── 1. % Negativos por plataforma ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
plat_neg = (dfs.groupby('plat')
              .apply(lambda x: (x['stars'] <= 3).mean() * 100)
              .sort_values(ascending=True))
colors_bar = [COLORES['neg'] if v >= 50 else COLORES['warn'] if v >= 30 else COLORES['pos']
              for v in plat_neg.values]
bars = ax1.barh(plat_neg.index, plat_neg.values, color=colors_bar)
ax1.axvline(50, color='red', linestyle='--', alpha=0.5, label='Umbral crítico 50%')
ax1.set_xlabel('% reseñas insatisfechas (1-3⭐)')
ax1.set_title('% Insatisfacción por plataforma', fontweight='bold')
for bar, val in zip(bars, plat_neg.values):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}%', va='center', fontsize=9)
ax1.legend(fontsize=8)

# ── 2. Distribución de tipificación (solo fallas) ──────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
fallas = dfs[dfs['tip'] != 'Buen funcionamiento']['tip'].value_counts().head(7)
colors_tip = [COLORES['neg'] if 'fábrica' in t.lower() or 'medición' in t.lower()
              else COLORES['warn'] for t in fallas.index]
ax2.barh(fallas.index[::-1], fallas.values[::-1], color=colors_tip[::-1])
ax2.set_xlabel('Número de reseñas')
ax2.set_title('Mapa de problemas — top 7 fallas', fontweight='bold')
for i, (idx, val) in enumerate(zip(fallas.index[::-1], fallas.values[::-1])):
    ax2.text(val + 0.3, i, str(val), va='center', fontsize=9)

# ── 3. Tendencia mensual 2025-2026 ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :])
orden_mes = ['enero','febrero','marzo','abril','mayo','junio',
             'julio','agosto','septiembre','octubre','noviembre','diciembre']
dfs_trend = dfs[dfs['año'].isin([2025, 2026])].copy()
dfs_trend['mes_ord'] = pd.Categorical(dfs_trend['mes_str'],
                                       categories=orden_mes, ordered=True)
dfs_trend['periodo'] = dfs_trend['año'].astype(int).astype(str) + '-' + \
                       dfs_trend['mes_ord'].astype(str)

tend = (dfs_trend.groupby(['año','mes_ord'], observed=True)
        .agg(n=('stars','count'),
             avg=('stars','mean'),
             pct_neg=('stars', lambda x: (x<=3).mean()*100))
        .reset_index()
        .sort_values(['año','mes_ord']))

tend['label'] = tend['mes_ord'].astype(str).str[:3] + ' ' + tend['año'].astype(int).astype(str).str[-2:]
x = range(len(tend))

ax3b = ax3.twinx()
ax3.bar(x, tend['pct_neg'], alpha=0.35, color=COLORES['neg'], label='% Insatisfechos (eje der.)')
ax3b.plot(x, tend['avg'], 'o-', color='#2c3e50', linewidth=2.5,
          markersize=6, label='Promedio ⭐ (eje izq.)')
ax3b.axhline(4.0, color=COLORES['pos'],  linestyle='--', alpha=0.6, label='Meta mínima 4.0⭐')
ax3b.axhline(3.44,color=COLORES['warn'], linestyle=':',  alpha=0.6, label=f'Actual promedio 3.44⭐')
ax3.set_xticks(list(x)); ax3.set_xticklabels(tend['label'], rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('% Insatisfechos', color=COLORES['neg'])
ax3b.set_ylabel('Promedio estrellas', color='#2c3e50')
ax3b.set_ylim(0, 5.5)
ax3.set_title('Tendencia mensual — % insatisfacción y promedio de estrellas (2025-2026)',
              fontweight='bold')
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

# ── 4. Promedio por plataforma vs meta ──────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
plat_avg = dfs.groupby('plat')['stars'].mean().sort_values()
colors_avg = [COLORES['neg'] if v < 3.0 else COLORES['warn'] if v < 4.0
              else COLORES['pos'] for v in plat_avg.values]
bars4 = ax4.barh(plat_avg.index, plat_avg.values, color=colors_avg)
ax4.axvline(4.0, color='green', linestyle='--', alpha=0.6, label='Meta 4.0⭐')
ax4.axvline(3.44, color='orange', linestyle=':', alpha=0.6, label='Promedio actual')
ax4.set_xlim(0, 5.2)
ax4.set_xlabel('Promedio de estrellas')
ax4.set_title('Promedio de estrellas por plataforma', fontweight='bold')
for bar, val in zip(bars4, plat_avg.values):
    ax4.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9)
ax4.legend(fontsize=8)

# ── 5. Distribución de estrellas global ─────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
star_cnt = dfs['stars'].value_counts().sort_index()
star_colors = [COLORES['neg'], COLORES['neg'], COLORES['warn'],
               COLORES['pos'], COLORES['pos']]
ax5.bar([f'{int(s)}⭐' for s in star_cnt.index],
        star_cnt.values, color=star_colors, edgecolor='white', linewidth=0.8)
for i, (s, v) in enumerate(zip(star_cnt.index, star_cnt.values)):
    pct = v / star_cnt.sum() * 100
    ax5.text(i, v + 2, f'{v}\n({pct:.0f}%)', ha='center', fontsize=9)
ax5.set_ylabel('Número de reseñas')
ax5.set_title('Distribución global de estrellas', fontweight='bold')
ax5.set_ylim(0, star_cnt.max() * 1.2)

plt.suptitle('Dashboard de Situación — Rotoplas IoT (2025-2026)', 
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('situacion_rotoplas_iot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard guardado como situacion_rotoplas_iot.png')


### Diagnóstico ejecutivo — ¿Estamos bien o mal?

**Estamos mal, y la tendencia está empeorando.**

| Indicador | Valor actual | Meta mínima | Semáforo |
|-----------|:-----------:|:-----------:|:--------:|
| Promedio global de estrellas | **3.44 / 5** | 4.0 | 🔴 |
| % Insatisfechos (1-3⭐) | **40.1%** | < 20% | 🔴 |
| App (Google Play) | **2.27 ⭐ — 70% negativo** | 4.0 | 🔴 |
| App (App Store) | **2.42 ⭐ — 68% negativo** | 4.0 | 🔴 |
| Kit Solar Amazon | **3.19 ⭐ — 47% negativo** | 4.0 | 🔴 |
| Kit Alámbrico Amazon | **3.64 ⭐ — 35% negativo** | 4.0 | 🟡 |
| Medidor Solar MeLi | **4.12 ⭐ — 23% negativo** | 4.0 | ✅ |
| Tendencia 2026 | Empeorando mes a mes | Estable | 🔴 |

#### Los 3 hallazgos más críticos para el negocio

**1. La app es el mayor riesgo de reputación** — 70% de reseñas negativas en Google Play y 68% en App Store. Un producto físico de IoT que falla en su interfaz digital pierde toda su propuesta de valor. Los clientes con el hardware funcionando igual dejan 1-2 estrellas por la app.

**2. El Kit Solar está en zona de crisis** — 47% de insatisfacción en Amazon. La categoría `Defecto de fábrica` concentra el mayor volumen: los productos fallan entre los 3-6 meses de uso. Esto genera el patrón más dañino para la marca: reseñas de clientes que *empezaron satisfechos* y cambiaron de opinión, lo cual demuestra que el problema no es de expectativas sino de calidad.

**3. La tendencia 2026 es de deterioro** — octubre-noviembre 2025 ya marcaron 83% de insatisfacción mensual. En 2026 ningún mes ha bajado del 56% de insatisfacción. Si no hay intervención, el promedio global seguirá cayendo por debajo de 3.0.